# Week 4 - Cross-Lingual Transfer (RQ1)

## Overview
This notebook investigates whether LaBSE can transfer semantic
relatedness understanding to Hausa without seeing any Hausa
training data (zero-shot transfer).

**Research Question 1:** Can a multilingual model trained only
on English data predict Hausa semantic relatedness?

## Step 1 - Install and Import Libraries
The following libraries are required to run this notebook.
- transformers: provides access to LaBSE from HuggingFace
- torch: runs and trains the model
- scipy: calculates Spearman correlation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from torch import nn
from torch.utils.data import DataLoader, Dataset
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

## Step 2 - Load Data
Loading the English training set (5,500 pairs) for fine-tuning
and the Hausa test set for zero-shot evaluation.

In [ ]:
#loading github repo
ENG_TRAIN="https://raw.githubusercontent.com/Rosemary2301/SemRel-Hausa-Project/refs/heads/main/data/english_train.csv"
HAS_TEST="https://raw.githubusercontent.com/Rosemary2301/SemRel-Hausa-Project/refs/heads/main/data/hausa_test.csv"



dfEngTrain = pd.read_csv(ENG_TRAIN)
dfHasTest= pd.read_csv(HAS_TEST)

## Step 3 - Dataset Class
Converts the dataframe into a format PyTorch can work with.
Each row returns the tokenized sentence pair and its similarity score.

In [ ]:
#gets data in form for training and testing
class SemRelDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx] #gets one sentence pair
        encoding =self.tokenizer(
            row['sentence1'],
            row['sentence2'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(row['label'], dtype=torch.float)
        }


## Step 4 - Model Class
Builds LaBSE with a regression head on top.
The regression head takes LaBSE's output and squashes
it to a single score between 0 and 1.

In [ ]:
class SemRelModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        #downloading pretrained model
        self.encoder = AutoModel.from_pretrained(model_name)
        #adding reegression head
        self.regressor = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        score=torch.sigmoid(self.regressor(cls_output))
        return score.squeeze()

## Step 5 - Fine-Tune LaBSE on English Data
Training LaBSE on the English SemRel training set for 3 epochs.
The loss should decrease with each epoch confirming the model is learning.

In [ ]:
modelName="sentence-transformers/LaBSE"
tokenizer= AutoTokenizer.from_pretrained(modelName)
model = SemRelModel(modelName)

trainDataset = SemRelDataset(dfEngTrain, tokenizer)
trainLoader=DataLoader(trainDataset, batch_size=16, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
lossFunc=nn.MSELoss()

#check to see if GPU available otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

#training loop
EPOCHS=3
print("Starting training...")
for epoch in range(EPOCHS):
    model.train() #model in training mode
    total_loss=0
    for batch in trainLoader:
        input_ids=batch['input_ids'].to(device)
        attention_mask=batch['attention_mask'].to(device)
        labels=batch['label'].to(device)

        optimizer.zero_grad()
        outputs=model(input_ids, attention_mask)
        loss=lossFunc(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(trainLoader):.4f}")

# Save the fine-tuned model
torch.save(model.state_dict(), "labse_english_finetuned.pt")
print("Model saved.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting training...
Epoch 1 | Loss: 0.0238
Epoch 2 | Loss: 0.0115
Epoch 3 | Loss: 0.0075
Model saved.


## Step 6 - Zero-Shot Testing on Hausa
Running the English-trained LaBSE directly on the Hausa test set.
The model has never seen Hausa — this tests whether its multilingual
knowledge transfers across languages automatically.

In [ ]:
def getPrediction(model,tokenizer,df,device,batchSize=16):
    dataset=SemRelDataset(df, tokenizer)
    loader=DataLoader(dataset, batch_size=batchSize)
    model.eval()
    predictions=[]

    with torch.no_grad():
        for batch in loader:
            input_ids=batch['input_ids'].to(device)
            attention_mask=batch['attention_mask'].to(device)
            pred=model(input_ids, attention_mask)
            predictions.extend(pred.cpu().numpy())

    return np.array(predictions)
hausaPreds=getPrediction(model,tokenizer,dfHasTest,device)

## Step 7 - Spearman Correlation
Evaluating how well the model's predictions rank against
the human gold labels using Spearman correlation.

In [ ]:
goldLabels=dfHasTest['label'].values
correlation,pVal=spearmanr(hausaPreds, goldLabels)
print(f"Spearman Correlation: {correlation:.4f} | p-value: {pVal:.4e}")

Spearman Correlation: 0.6509 | p-value: 6.1680e-74


## Results



In [3]:
import pandas as pd
results_df = pd.DataFrame({
    "Model": [
        "Dice Baseline",
        "AfriBERTa",
        "mBERT",
        "LaBSE Zero-Shot"
    ],
    "Spearman Correlation": [
        0.4031,
        0.4974,
        0.5947,
        0.6509
    ]
})

results_df

,Model,Spearman Correlation
0,Dice Baseline,0.4031
1,AfriBERTa,0.4974
2,mBERT,0.5947
3,LaBSE Zero-Shot,0.6509
